# Сравнение различных подходов к Voice Activity Detection (VAD)
В этом ноутбуке мы реализуем и сравним 4 подхода к детектированию речи на датасете RAVDESS:
1. **MLP** (Многослойный перцептрон) на признаках MFCC
2. **LSTM** (Рекуррентная сеть) для учета временного контекста
3. **1D-CNN** (Сверточная сеть) для извлечения паттернов из аудио
4. **MediaPipe** (Визуальный VAD) на основе расстояния между губами

In [1]:
import os
import cv2
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## 1. Подготовка данных RAVDESS

In [2]:
RAVDESS_PATH = "../datasets/orvile/ravdess-dataset/versions/1/"
MODEL_ASSET_PATH = "../raw_models/face_landmarker.task"

def get_paired_files(path, limit=100):
    """Находит пары аудио-видео файлов RAVDESS"""
    pairs = []
    # Ищем в Audio_Speech
    audio_base = os.path.join(path, "Audio_Speech_Actors_01-24")
    for actor_dir in sorted(os.listdir(audio_base)):
        if not actor_dir.startswith("Actor_"): continue
        audio_path = os.path.join(audio_base, actor_dir)
        for file in os.listdir(audio_path):
            if file.endswith(".wav"):
                # Генерируем имя видео файла
                # 03-01-... -> 01-01-...
                video_name = "01" + file[2:-4] + ".mp4"
                video_folder = "Video_Speech_" + actor_dir
                video_path = os.path.join(path, video_folder, actor_dir, video_name)
                
                if os.path.exists(video_path):
                    pairs.append({"audio": os.path.join(audio_path, file), "video": video_path})
                
                if len(pairs) >= limit: return pairs
    return pairs

paired_data = get_paired_files(RAVDESS_PATH, limit=50) # Для эксперимента ограничимся 50 парами
print(f"Найдено {len(paired_data)} пар аудио-видео.")

Найдено 50 пар аудио-видео.


## 2. Извлечение признаков

In [3]:
def extract_audio_features(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=16000)
    # Вычисляем MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=512)
    # Вычисляем RMS для автоматической разметки (ground truth)
    rms = librosa.feature.rms(y=y, hop_length=512)[0]
    labels = (rms > (np.max(rms) * 0.15)).astype(int)
    
    # Синхронизируем длину
    min_len = min(mfcc.shape[1], labels.shape[0])
    return mfcc[:, :min_len].T, labels[:min_len]

X_audio = []
y_labels = []
print("Извлечение аудио признаков...")
for pair in tqdm(paired_data):
    feats, labs = extract_audio_features(pair['audio'])
    X_audio.append(feats)
    y_labels.append(labs)

X_audio = np.vstack(X_audio)
y_labels = np.concatenate(y_labels)
print(f"X shape: {X_audio.shape}, y shape: {y_labels.shape}")

Извлечение аудио признаков...


  0%|          | 0/50 [00:00<?, ?it/s]/Users/kaparya/Desktop/Diploma/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 50/50 [00:12<00:00,  4.15it/s]

X shape: (5890, 13), y shape: (5890,)


## 3. Определения моделей

In [7]:
class MLP_VAD(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

class LSTM_VAD(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 32, batch_first=True, num_layers=1)
        self.fc = nn.Linear(32, 1)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        # x: [batch, seq, feat]
        out, _ = self.lstm(x)
        return self.sig(self.fc(out))

class CNN1D_VAD(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Для Conv1d input_dim — это количество входных каналов
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 1, kernel_size=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        # Если x приходит как (Batch, 1, 13), переставляем в (Batch, 13, 1)
        if x.dim() == 3 and x.shape[1] == 1:
            x = x.transpose(1, 2)
        return self.net(x).transpose(1, 2)

## 4. Обучение аудио моделей

In [8]:
# Подготовка данных
X_train, X_test, y_train, y_test = train_test_split(X_audio, y_labels, test_size=0.2, random_state=42)

def train_model(model, X, y, epochs=15, is_seq=False):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    y_t = torch.tensor(y, dtype=torch.float32).reshape(-1, 1).to(device)
    
    if is_seq:
        # Делаем форму (Batch, 1, 13) для последовательности из 1 кадра
        X_t = X_t.unsqueeze(1)
        y_t = y_t.unsqueeze(1)
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_t)
        loss = criterion(outputs, y_t)
        loss.backward()
        optimizer.step()
    return model

print("Обучение MLP...")
mlp = train_model(MLP_VAD(13), X_train, y_train)

print("Обучение LSTM...")
lstm = train_model(LSTM_VAD(13), X_train, y_train, is_seq=True)

print("Обучение CNN...")
cnn = train_model(CNN1D_VAD(13), X_train, y_train, is_seq=True)

Обучение MLP...
Обучение LSTM...
Обучение CNN...


: 

## 5. Сравнение результатов

In [ ]:
results = []

def evaluate(name, model, X, y, is_seq=False):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        if is_seq:
            X_t = X_t.unsqueeze(1)
            if isinstance(model, CNN1D_VAD):
                preds = model(X_t.transpose(1, 2)).transpose(1, 2).cpu().numpy().flatten()
            else:
                preds = model(X_t).cpu().numpy().flatten()
        else:
            preds = model(X_t).cpu().numpy().flatten()
    
    binary_preds = (preds > 0.5).astype(int)
    acc = accuracy_score(y, binary_preds)
    f1 = f1_score(y, binary_preds)
    results.append({"Model": name, "Accuracy": acc, "F1-Score": f1})

evaluate("MLP", mlp, X_test, y_test)
evaluate("LSTM", lstm, X_test, y_test, is_seq=True)
evaluate("1D-CNN", cnn, X_test, y_test, is_seq=True)

df_results = pd.DataFrame(results)
print(df_results)